# Negotiation: Household Budget Plan

A savings agent and lifestyle agent negotiate trade-offs and propose a balanced monthly plan.

This notebook calls an LLM using `langchain_openai`. Add `OPENAI_API_KEY` to `.env` before running.

## 1. Load the API key and connect to the model

`load_dotenv()` reads the key from `.env`. The key is not written in the notebook.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
MODEL_NAME = os.getenv('OPENAI_MODEL', 'gpt-4.1-mini')
llm = ChatOpenAI(model=MODEL_NAME, temperature=0.2)
print('Model ready:', MODEL_NAME)


## 2. Add a small helper for each agent

Each call has a role, a task, and the current context. This makes the handoff visible.

In [ ]:
def ask_agent(role, task, context):
    prompt = f"""You are the {role}.
Task: {task}
Input and prior agent messages:
{context}

Give a concise, practical answer. Separate facts from assumptions. Do not claim to have verified live information unless provided."""
    return llm.invoke(prompt).content


## External tool: convert_currency

Budget planning can use a current exchange rate when a budget includes a foreign-currency amount. Add amount/currency inputs and provide the fetched rate to the agents.

This tool makes a real web request. The returned data is then added to the agents’ context.


In [ ]:
import requests
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current weather and today's forecast for a city using Open-Meteo."""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1, "language": "en", "format": "json"}, timeout=15).json()
    places = geo.get("results", [])
    if not places:
        return f"No location found for {city}."
    place = places[0]
    weather = requests.get("https://api.open-meteo.com/v1/forecast", params={"latitude": place["latitude"], "longitude": place["longitude"], "current": "temperature_2m,precipitation,wind_speed_10m", "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max", "timezone": "auto", "forecast_days": 1}, timeout=15).json()
    return f"Source: Open-Meteo. Location: {place['name']}, {place.get('country','')}. Data: {weather}"

@tool
def convert_currency(amount: float, source_currency: str, target_currency: str) -> str:
    """Convert an amount using the latest available reference exchange rate."""
    data = requests.get(f"https://api.frankfurter.dev/v1/latest", params={"base": source_currency.upper(), "symbols": target_currency.upper()}, timeout=15).json()
    rate = data.get("rates", {}).get(target_currency.upper())
    if rate is None:
        return f"No exchange rate returned for {source_currency} to {target_currency}."
    return f"Source: Frankfurter. Rate date: {data.get('date')}. {amount:.2f} {source_currency.upper()} = {amount*rate:.2f} {target_currency.upper()} (rate {rate})."

def call_tool(tool_function, **inputs):
    """Run a real API-backed LangChain tool and return its result."""
    return tool_function.invoke(inputs)



## 3. Provide the use-case request

In [ ]:
user_request = 'Monthly discretionary budget is ₹20,000. Save more while keeping one family outing.'
context = user_request
print(user_request)


In [ ]:
amount = 1000
source_currency = 'USD'
target_currency = 'INR'
live_tool_result = call_tool(convert_currency, amount=amount, source_currency=source_currency, target_currency=target_currency)
print(live_tool_result)
context += '\n\nExternal tool result (Frankfurter): ' + str(live_tool_result)


## 4. Run agents and pass their messages along

For each step, the agent sees the original request and any earlier messages. In a pattern with independent agents, each agent gets only the original request.

In [ ]:
workflow = ['Savings Agent: suggest a realistic savings amount and explain trade-offs.', 'Lifestyle Agent: protect one family outing while reviewing that proposal.', 'Mediator Agent: find a balanced allocation that totals ₹20,000.']
for instruction in workflow:
    role, task = instruction.split(':', 1)
    answer = ask_agent(role, task.strip(), context)
    print(f'\n### {role}\n{answer}')
    context += f'\n\n{role} said:\n{answer}'


## 5. Review

This is a live LLM demonstration, not a guarantee of correctness. Verify recommendations. The example does not take external actions such as purchases, bookings, access changes, or production updates.